In [ ]:
"""
Time Series — 4 Stacked Subplots
(a) Niño 3.4 | DMI              (grouped bars)
(b) SST (line) | CHL (bars)
(c) CHL & SST (lines, twin axes) vs Niño 3.4 (bars, right axis)
(d) CHL & SST (lines, twin axes) vs DMI (bars, right axis)
Revised: 282-month calendar with gaps shown as gaps, larger labels,
legends INSIDE each panel, units match processed data.
Data: deseasonalized CHL, detrended SST, original Niño 3.4 and DMI.
All series: 4-month centered running mean (calculated on real calendar months).
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

# ─────────────────────────────────────────────
# USER INPUTS
# ─────────────────────────────────────────────
INPUT_FILE  = "Anomalized_detrended_SST&CHL.xlsx"
OUTPUT_PATH = r"Timeseries_revised.pdf"
# ─────────────────────────────────────────────

# ── Labels and units: edit ONLY here ──────────
LBL_NINO = "Niño 3.4"
LBL_DMI  = "DMI"
LBL_SST  = "SST Anomaly (°C)"
LBL_CHL  = r"CHL Anomaly (mg m$^{-3}$)"

# ── Font sizes ────────────────────────────────
FS_BASE   = 16
FS_LABEL  = 17
FS_TICK   = 14
FS_XTICK  = 13
FS_LEGEND = 14
FS_PANEL  = 20
LW        = 2.4

# Headroom factor: upper limit = K * lower limit magnitude.
# K = 1.5 leaves a blank band of 20% of the axis height above the data.
# If a legend still touches data, raise K (for example 1.7).
K_HEAD = 1.5

# ── Load data ────────────────────────────────
df = pd.read_excel(INPUT_FILE, sheet_name="Detrended_Data")
df.columns = df.columns.str.replace(r"\s+", " ", regex=True).str.strip()

COL_TIME = "Month-Year"
COL_NINO = "Niño 3.4"
COL_DMI  = "DMI"
COL_SST  = "SST (°C)"
COL_CHL  = "CHL (mg/m^3)"
VARS = [COL_NINO, COL_DMI, COL_SST, COL_CHL]

df["Date"] = pd.to_datetime(df[COL_TIME], format="%b-%Y")
df = df.sort_values("Date").set_index("Date")[VARS].copy()
for c in VARS:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Full monthly calendar: missing months become NaN rows (no dropna!)
df = df.asfreq("MS")

print(f"Calendar months: {len(df)}  ({df.index.min():%b-%Y} to {df.index.max():%b-%Y})")
print("Missing months per variable:")
print(df.isna().sum().to_string())

# ── 4-month centered running mean on the calendar ──
smooth = df.rolling(window=4, center=True, min_periods=2).mean()
smooth = smooth.where(df.notna())   # no smoothed values in months with no data

time = smooth.index.values
nino = smooth[COL_NINO].values
dmi  = smooth[COL_DMI].values
sst  = smooth[COL_SST].values
chl  = smooth[COL_CHL].values
x    = np.arange(len(time))          # now 1 step = 1 real month
n_ticks = 24                         # one tick every 2 years (use 12 for yearly)

# ── Style ─────────────────────────────────────
plt.rcParams.update({
    "font.family":         "serif",
    "font.serif":          ["Times New Roman", "DejaVu Serif"],
    "font.size":           FS_BASE,
    "font.weight":         "bold",
    "axes.labelsize":      FS_LABEL,
    "axes.labelweight":    "bold",
    "axes.titlesize":      FS_LABEL,
    "axes.titleweight":    "bold",
    "xtick.labelsize":     FS_TICK,
    "ytick.labelsize":     FS_TICK,
    "axes.linewidth":      1.5,
    "xtick.direction":     "in",
    "ytick.direction":     "in",
    "xtick.major.size":    5,
    "ytick.major.size":    5,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "figure.dpi":          150,
    "savefig.dpi":         600,
    "mathtext.fontset":    "custom",
    "mathtext.rm":         "Times New Roman",
    "mathtext.it":         "Times New Roman:italic",
    "mathtext.bf":         "Times New Roman:bold",
    "pdf.fonttype":        42,
    "ps.fonttype":         42,
})

# Colors (unchanged)
C_NINO = "#74add1"
C_DMI  = "#7b2d2d"
C_SST  = "#d73027"
C_CHL  = "#1a9641"

# ── Helpers ───────────────────────────────────
def set_xaxis(ax):
    ax.set_xticks(x[::n_ticks])
    ax.set_xticklabels(
        pd.DatetimeIndex(time[::n_ticks]).strftime("%b\n%Y"),
        fontsize=FS_XTICK, ha="center"
    )
    ax.set_xlim(-0.8, len(x) - 0.2)

def panel_label(ax, label):
    ax.text(0.01, 0.97, label, transform=ax.transAxes,
            fontsize=FS_PANEL, fontweight="bold", va="top", ha="left",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      edgecolor="black", linewidth=1.0, alpha=0.95))

def set_ylim_headroom(ax, arr, k=K_HEAD, step=0.5, pad=0.15):
    """
    Lower limit = -L, upper limit = k*L. The extra space on top is the
    blank band for the legend. The same k on every axis keeps the zero
    lines aligned across twin axes. Ticks cover only the data range.
    NaN-safe (missing months are ignored).
    """
    mx = np.nanmax(np.abs(arr))
    mx = np.ceil(mx / step) * step
    L = mx + pad
    ax.set_ylim(-L, k * L)
    ax.set_yticks(np.arange(-mx, mx + step / 2, step))

def legend_inside(ax, handles, ncol):
    """Legend inside the axes, in the blank band at the top centre."""
    ax.legend(handles=handles, loc="upper center",
              bbox_to_anchor=(0.5, 0.985), ncol=ncol,
              fontsize=FS_LEGEND, frameon=True, framealpha=1.0,
              edgecolor="black", columnspacing=1.6,
              handlelength=2.2, handleheight=1.1, borderpad=0.5)

def line_h(color, label):
    return mlines.Line2D([], [], color=color, linewidth=LW + 0.4, label=label)

def bar_h(color, label):
    return mpatches.Patch(facecolor=color, edgecolor="none", alpha=0.85, label=label)

# ── Figure ────────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(14, 20), constrained_layout=False)
plt.subplots_adjust(left=0.18, right=0.89, top=0.97, bottom=0.05, hspace=0.30)

bar_width = 0.70

# ════════ Panel (a) — Niño 3.4 + DMI bars ════════
ax1 = axes[0]
ax1.bar(x - bar_width / 2, nino, width=bar_width,
        color=C_NINO, alpha=0.85, linewidth=0, zorder=2)
ax1.bar(x + bar_width / 2, dmi, width=bar_width,
        color=C_DMI, alpha=0.85, linewidth=0, zorder=2)
ax1.axhline(0, color="black", linewidth=0.9, zorder=4)

ax1.set_ylabel("Niño 3.4 & DMI (°C)", labelpad=4)
ax1.yaxis.set_label_coords(-0.07, 0.5)
set_xaxis(ax1)
set_ylim_headroom(ax1, np.r_[nino, dmi])
ax1.tick_params(which="both", top=True, right=True)
legend_inside(ax1, [bar_h(C_NINO, LBL_NINO), bar_h(C_DMI, LBL_DMI)], ncol=2)
panel_label(ax1, "(a)")

# ════════ Panel (b) — SST line + CHL bars ════════
ax2 = axes[1]
ax2.plot(x, sst, color=C_SST, linewidth=LW, linestyle="-", zorder=3)   # line breaks at gaps
ax2.bar(x + bar_width / 2, chl, width=bar_width,
        color=C_CHL, alpha=0.85, linewidth=0, zorder=2)
ax2.axhline(0, color="black", linewidth=0.9, zorder=1)

ax2.set_ylabel("SST Anomaly (°C)\n" + r"CHL Anomaly (mg m$^{-3}$)", labelpad=4)
ax2.yaxis.set_label_coords(-0.07, 0.5)
set_xaxis(ax2)
set_ylim_headroom(ax2, np.r_[sst, chl])
ax2.tick_params(which="both", top=True, right=True)
legend_inside(ax2, [line_h(C_SST, LBL_SST), bar_h(C_CHL, LBL_CHL)], ncol=2)
panel_label(ax2, "(b)")

# ════════ Panels (c), (d) ════════
def build_panel(base_ax, bar_data, bar_color, bar_label, right_label, panel_id):

    # CHL — left axis
    ax_chl = base_ax
    ax_chl.set_zorder(3)
    ax_chl.patch.set_visible(False)
    ax_chl.plot(x, chl, color=C_CHL, linewidth=LW, linestyle="-", zorder=3)
    ax_chl.set_ylabel(LBL_CHL, labelpad=4)
    ax_chl.yaxis.set_label_coords(-0.055, 0.5)
    ax_chl.tick_params(axis="y", which="both", colors=C_CHL, direction="in")
    ax_chl.spines["left"].set_color(C_CHL)
    ax_chl.yaxis.label.set_color(C_CHL)
    ax_chl.spines["right"].set_visible(False)
    set_xaxis(ax_chl)

    # SST — second left axis (shifted out)
    ax_sst = ax_chl.twinx()
    ax_sst.set_zorder(2)
    ax_sst.patch.set_visible(False)
    ax_sst.plot(x, sst, color=C_SST, linewidth=LW, linestyle="-", zorder=2)
    ax_sst.set_ylabel(LBL_SST, labelpad=4)
    ax_sst.yaxis.set_label_position("left")
    ax_sst.yaxis.tick_left()
    ax_sst.spines["left"].set_position(("axes", -0.115))
    ax_sst.spines["left"].set_color(C_SST)
    ax_sst.spines["left"].set_visible(True)
    ax_sst.spines["right"].set_visible(False)
    ax_sst.tick_params(axis="y", which="both", colors=C_SST, direction="in")
    ax_sst.yaxis.label.set_color(C_SST)
    ax_sst.set_xlim(-0.8, len(x) - 0.2)

    # Niño / DMI bars — right axis
    ax_bar = ax_chl.twinx()
    ax_bar.set_zorder(1)
    ax_bar.patch.set_visible(True)
    ax_bar.bar(x, bar_data, width=0.7, color=bar_color,
               alpha=0.85, linewidth=0, zorder=1)
    ax_bar.axhline(0, color="black", linewidth=0.9, zorder=0)
    ax_bar.set_xlim(-0.8, len(x) - 0.2)
    ax_bar.set_ylabel(right_label, labelpad=8)
    ax_bar.spines["right"].set_color(bar_color)
    ax_bar.tick_params(axis="y", which="both", colors=bar_color, direction="in")
    ax_bar.yaxis.label.set_color(bar_color)
    ax_bar.spines["left"].set_visible(False)

    ax_chl.axhline(0, color="black", linewidth=0.8, linestyle="--",
                   alpha=0.4, zorder=0)

    # Same headroom factor on all three axes -> zero lines stay aligned
    set_ylim_headroom(ax_chl, chl)
    set_ylim_headroom(ax_sst, sst)
    set_ylim_headroom(ax_bar, bar_data)

    ax_sst.tick_params(axis="x", which="both", bottom=False, labelbottom=False)
    ax_bar.tick_params(axis="x", which="both", bottom=False, labelbottom=False)

    legend_inside(ax_chl,
                  [bar_h(bar_color, bar_label),
                   line_h(C_SST, LBL_SST),
                   line_h(C_CHL, LBL_CHL)],
                  ncol=3)
    panel_label(ax_chl, panel_id)

build_panel(axes[2], nino, C_NINO, LBL_NINO, LBL_NINO, "(c)")
build_panel(axes[3], dmi,  C_DMI,  LBL_DMI,  LBL_DMI,  "(d)")

# ── Save & show ───────────────────────────────
fig.savefig(OUTPUT_PATH, format="pdf", bbox_inches="tight", facecolor="white")
print(f"Figure saved → {OUTPUT_PATH}")
plt.show()